# Klasifikasi teks MULTI-LABEL (v2): ujaran kebencian & bahasa kasar Twitter Indonesia

**Butuh:** `train` (kolom `Tweet` + 12 label 0/1), `test` (kolom `id` + `Tweet`).
**Menghasilkan:** `submission.csv` (`id` + 12 kolom 0/1), prediksi out-of-fold & test tiap model di
`outputs_cv/`, laporan F1 per label.

## Beda dengan v1
| v1 | v2 |
|---|---|
| satu split 80/20; epoch, ambang, bobot disetel di 2.107 baris | **5-fold CV**: ambang & bobot disetel di prediksi out-of-fold (OOF) atas **seluruh** 10.533 baris |
| prediksi test dari model yang dilatih di 80% train | prediksi test = **rata-rata 5 model fold** (tiap model melihat 80% data berbeda) |
| satu backbone BERT | **IndoBERT + IndoBERTweet** (model khusus Twitter Indonesia) + TF-IDF |
| epoch terbaik dipilih di val (bocor tipis ke OOF) | epoch tetap (`EPOCHS`), ambil epoch terakhir → OOF tidak bias (bisa diubah) |
| embedding MiniLM selalu jalan | opsional, default mati (lemah: F1 test ±0,51) |
| peringatan `id2label` panjang 5 | `id2label`/`label2id` 12 label diberikan eksplisit |

Struktur data dan metrik sama seperti v1: 12 label berhierarki (`HS=0` ⇒ semua `HS_*` = 0; `HS=1` ⇒
tepat satu target dan satu tingkat), metrik **F1 macro** atas 12 label, ambang per label.

## Cara menempel / menjalankan

| sel | isi | wajib? |
|---|---|---|
| 0 | data (pola template: `train`, `test`, `test_labels`) | **ya** |
| 1 | konfigurasi + 5 fold tetap (dipakai SEMUA model) | **ya** |
| 2 | fungsi bantu (F1, ambang, hierarki, CV) | **ya** |
| 3 | lihat data singkat | opsional |
| 4 | TF-IDF + leksikon kata kasar, 5-fold (±2–3 menit CPU) | **ya** |
| 5 | **unduh IndoBERTweet** sekali (±450 MB) | sekali, kalau pakai IndoBERTweet |
| 6 | embedding MiniLM, 5-fold | opsional (default mati) |
| 7a–7b | fine-tune BERT 5-fold per backbone (GPU) | paling akurat |
| 8 | gabung semua → bobot, ambang, dekode → `submission.csv` | **ya** |
| 9 | cek ke kunci jawaban (hanya demo) | opsional |

Waktu BERT ≈ (waktu 1 kali latih di v1) × `N_FOLDS` × jumlah backbone × jumlah seed. Hasil tiap
fold disimpan, jadi kalau kernel mati atau dihentikan, jalankan ulang sel 7b dan ia melanjutkan.
Terlalu lama? Turunkan `N_FOLDS` ke 3 atau isi `BERT_MODELS = ["indobertweet"]`.

## 0. Data (pola sama dengan template; isi tidak diubah)

In [1]:
import pandas as pd

DEMO_FOLDER = "."   # folder data_latihan (di sebelah repo)

train = pd.read_csv(f"hs_train.csv")
test = pd.read_csv(f"hs_test.csv")
test_labels = pd.read_csv(f"hs_test_labels.csv")
sample_submission = pd.read_csv(f"hs_sample_submission.csv")
print(train.shape, test.shape, sample_submission.shape)

(10533, 14) (2636, 2) (2636, 13)


## 1. Konfigurasi + 5 fold tetap
**Menghasilkan:** `LABELS`, `Y`, `FOLDS` (daftar `(index_train, index_val)`), indeks hierarki.
Fold dibuat sekali (stratifikasi menurut kombinasi 12 label; kombinasi < 20 baris digabung; tweet
kembar persis dijaga satu sisi) dan dipakai semua model, sehingga OOF mereka sejajar baris-per-baris.

`find_pretrained_folder` mencari folder `pretrained` dari folder kerja naik ke folder induk;
`model_path(nama)` memberi pesan jelas kalau modelnya tidak ada.

In [2]:
import os
import re
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import StratifiedGroupKFold

TEXT = "Tweet"                          # UBAH
LABELS = [c for c in train.columns if c not in ("id", TEXT)]
assert LABELS == [c for c in test_labels.columns if c != "id"] == [c for c in sample_submission.columns if c != "id"]
RESOURCE_FOLDER = f"{DEMO_FOLDER}"   # abusive.csv + new_kamusalay.csv
CV_FOLDER = "outputs_cv"               # hasil CV tiap model (terpisah dari outputs/ milik v1)
N_FOLDS = 5                            # UBAH: 3 kalau BERT terlalu lama
BERT_MODELS = ["indobert", "indobertweet"]   # UBAH: salah satu saja kalau waktu terbatas
RUN_EMBEDDING = False                  # UBAH: True kalau ingin ikut mencoba MiniLM
RUN_BERT = True                        # UBAH: False kalau tidak ada GPU
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CV_FOLDER, exist_ok=True)


def find_pretrained_folder(name="pretrained"):
    # cari folder `pretrained` di folder kerja, lalu naik ke folder induk (mis. root project)
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / name).is_dir():
            return str(folder / name)
    return name


PRETRAINED_FOLDER = r"C:\INAH-MINE\GEMASTIK-XIX\pretrained"   # UBAH: isi manual kalau di tempat lain, mis. r"D:\models\pretrained"
print("folder pretrained:", PRETRAINED_FOLDER)


def model_path(name):
    path = Path(PRETRAINED_FOLDER) / name
    if not path.is_dir():
        raise FileNotFoundError(f"'{name}' tidak ada di {path}. Isi PRETRAINED_FOLDER dengan "
                                f"folder yang berisi '{name}'.")
    return str(path)


train[TEXT] = train[TEXT].fillna("").astype(str)
test[TEXT] = test[TEXT].fillna("").astype(str)

# struktur hierarki label
HS_IDX = LABELS.index("HS")
SUB_IDX = [i for i, name in enumerate(LABELS) if name.startswith("HS_")]
TYPE_IDX = [LABELS.index(n) for n in ("HS_Individual", "HS_Group")]
LEVEL_IDX = [LABELS.index(n) for n in ("HS_Weak", "HS_Moderate", "HS_Strong")]
CATEGORY_IDX = [LABELS.index(n) for n in ("HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", "HS_Other")]
ID2LABEL = dict(enumerate(LABELS))
LABEL2ID = {name: i for i, name in enumerate(LABELS)}

Y = train[LABELS].to_numpy().astype(int)
combo = pd.Series(["".join(map(str, row)) for row in Y])
strat_key = np.where(combo.map(combo.value_counts()) >= 20, combo, "jarang")
splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(splitter.split(Y, strat_key, groups=train[TEXT]))
print(len(LABELS), "label |", N_FOLDS, "fold, ukuran val:", [len(va) for _, va in FOLDS], "| device:", DEVICE)

folder pretrained: C:\INAH-MINE\GEMASTIK-XIX\pretrained
12 label | 5 fold, ukuran val: [2107, 2107, 2106, 2106, 2107] | device: cuda


## 2. Fungsi bantu
- `f1_per_label`, `macro_f1`, `tune_thresholds`, `decode_hierarchy`, `report`: sama seperti v1.
- `fit_predict_proba`: satu model untuk 12 label; `conditional=True` menambah model sub-label yang
  hanya dilatih pada tweet `HS=1`, lalu `P(sub) = P(HS) × P(sub | HS)`.
- `run_cv` (model sklearn) dan `run_cv_neural` (model neural, dengan cache per fold/seed): hasilnya
  **OOF** (tiap baris train diprediksi oleh model yang tidak melihatnya) dan **test = rata-rata fold**.

In [3]:
GRID = np.arange(0.10, 0.91, 0.05)


def f1_per_label(y_true, y_bin):
    tp = (y_true * y_bin).sum(axis=0)
    fp = ((1 - y_true) * y_bin).sum(axis=0)
    fn = (y_true * (1 - y_bin)).sum(axis=0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def macro_f1(y_true, y_bin):
    return f1_per_label(y_true, y_bin).mean()


def tune_thresholds(y_true, proba):
    scores = np.array([f1_per_label(y_true, (proba >= g).astype(int)) for g in GRID])   # (ambang, label)
    return GRID[scores.argmax(axis=0)]


def decode_hierarchy(proba, thresholds):
    binary = (proba >= thresholds).astype(int)
    is_hs = binary[:, HS_IDX] == 1
    binary[np.ix_(~is_hs, SUB_IDX)] = 0                      # HS=0 -> semua sub-label 0
    for group in (TYPE_IDX, LEVEL_IDX):                      # tepat satu per kelompok
        chosen = np.eye(len(group), dtype=int)[proba[:, group].argmax(axis=1)]
        binary[np.ix_(is_hs, group)] = chosen[is_hs]
    no_category = is_hs & (binary[:, CATEGORY_IDX].sum(axis=1) == 0)   # minimal satu kategori
    best = np.array(CATEGORY_IDX)[proba[:, CATEGORY_IDX].argmax(axis=1)]
    binary[no_category, best[no_category]] = 1
    return binary


def report(y_true, y_bin):
    f1 = f1_per_label(y_true, y_bin)
    return pd.DataFrame({"jumlah_positif": y_true.sum(axis=0), "diprediksi": y_bin.sum(axis=0),
                         "F1": f1.round(4)}, index=LABELS)


def fit_predict_proba(make_estimator, X_fit, Y_fit, X_preds, conditional=True):
    model = make_estimator().fit(X_fit, Y_fit)
    probas = [model.predict_proba(X) for X in X_preds]
    if conditional:
        is_hs = Y_fit[:, HS_IDX] == 1
        sub_model = make_estimator().fit(X_fit[is_hs], Y_fit[is_hs][:, SUB_IDX])
        for X, proba in zip(X_preds, probas):
            proba[:, SUB_IDX] = proba[:, [HS_IDX]] * sub_model.predict_proba(X)
    return probas


def save_cv(name, oof, test_proba):
    np.save(f"{CV_FOLDER}/oof_{name}.npy", oof)
    np.save(f"{CV_FOLDER}/test_{name}.npy", test_proba)
    print(f"[{name}] F1 macro OOF (ambang per label): "
          f"{macro_f1(Y, (oof >= tune_thresholds(Y, oof)).astype(int)):.4f} | tersimpan di {CV_FOLDER}/")


def run_cv(name, make_estimator, X, X_test, conditional=True):
    oof = np.zeros((len(Y), len(LABELS)))
    test_sum = np.zeros((len(X_test), len(LABELS)))
    for fold, (index_train, index_val) in enumerate(FOLDS):
        started = time.time()
        val_proba, test_proba = fit_predict_proba(make_estimator, X[index_train], Y[index_train],
                                                  [X[index_val], X_test], conditional)
        oof[index_val] = val_proba
        test_sum += test_proba / len(FOLDS)
        print(f"[{name}] fold {fold + 1}/{len(FOLDS)}: F1 val (ambang 0,5) "
              f"{macro_f1(Y[index_val], (val_proba >= 0.5).astype(int)):.4f} ({time.time() - started:.0f} dtk)")
    save_cv(name, oof, test_sum)
    return oof, test_sum


def run_cv_neural(name, fold_function, seeds=(42,), force=False):
    # fold_function(index_train, index_val, seed) -> (proba val, proba test); hasil tiap fold/seed di-cache
    oof = np.zeros((len(Y), len(LABELS)))
    test_sum = np.zeros((len(test), len(LABELS)))
    for fold, (index_train, index_val) in enumerate(FOLDS):
        fold_val, fold_test = [], []
        for seed in seeds:
            cache = f"{CV_FOLDER}/{name}_fold{fold}_seed{seed}_of{len(FOLDS)}.npz"
            if os.path.exists(cache) and not force:
                saved = np.load(cache)
                val_proba, test_proba = saved["val"], saved["test"]
                print(f"[{name}] fold {fold + 1} seed {seed}: pakai hasil tersimpan")
            else:
                started = time.time()
                val_proba, test_proba = fold_function(index_train, index_val, seed)
                np.savez(cache, val=val_proba, test=test_proba)
                print(f"[{name}] fold {fold + 1} seed {seed}: selesai ({(time.time() - started) / 60:.1f} menit)")
            fold_val.append(val_proba)
            fold_test.append(test_proba)
        oof[index_val] = np.mean(fold_val, axis=0)
        test_sum += np.mean(fold_test, axis=0) / len(FOLDS)
        print(f"[{name}] fold {fold + 1}/{len(FOLDS)}: F1 val (ambang 0,5) "
              f"{macro_f1(Y[index_val], (oof[index_val] >= 0.5).astype(int)):.4f}")
    save_cv(name, oof, test_sum)
    return oof, test_sum


print("fungsi siap; 12 label, sub-label HS_*:", len(SUB_IDX))

fungsi siap; 12 label, sub-label HS_*: 10


## 3. Lihat data singkat

In [4]:
print("HS=0 tetapi ada sub-label:", int(((Y[:, HS_IDX] == 0) & (Y[:, SUB_IDX].sum(axis=1) > 0)).sum()))
is_hs = Y[:, HS_IDX] == 1
print("HS=1: jumlah target (harus 1):", set(Y[is_hs][:, TYPE_IDX].sum(axis=1)),
      "| jumlah tingkat (harus 1):", set(Y[is_hs][:, LEVEL_IDX].sum(axis=1)),
      "| jumlah kategori:", sorted(set(Y[is_hs][:, CATEGORY_IDX].sum(axis=1))))
print("tweet kembar di train:", int(train.duplicated(TEXT).sum()),
      "| panjang kata median/95%:", np.percentile(train[TEXT].str.split().str.len(), [50, 95]))
print(pd.DataFrame({f"fold{k + 1}": Y[va].mean(axis=0) for k, (_, va) in enumerate(FOLDS)},
                   index=LABELS).round(3).T.to_string())

HS=0 tetapi ada sub-label: 0
HS=1: jumlah target (harus 1): {np.int64(1)} | jumlah tingkat (harus 1): {np.int64(1)} | jumlah kategori: [np.int64(1), np.int64(2), np.int64(3)]
tweet kembar di train: 117 | panjang kata median/95%: [15. 38.]
          HS  Abusive  HS_Individual  HS_Group  HS_Religion  HS_Race  HS_Physical  HS_Gender  HS_Other  HS_Weak  HS_Moderate  HS_Strong
fold1  0.422    0.380          0.271     0.151        0.059    0.043        0.025      0.021     0.284    0.255        0.129      0.037
fold2  0.422    0.385          0.274     0.148        0.062    0.044        0.024      0.022     0.285    0.260        0.128      0.035
fold3  0.422    0.381          0.273     0.149        0.059    0.042        0.025      0.024     0.284    0.257        0.129      0.036
fold4  0.422    0.385          0.271     0.151        0.060    0.042        0.025      0.024     0.285    0.257        0.130      0.035
fold5  0.422    0.384          0.271     0.151        0.062    0.041        0.026

## 4. TF-IDF + leksikon kata kasar + LogisticRegression, 5-fold
Sama seperti v1 (TF-IDF kata 1–2 + huruf 2–5 + 6 fitur dari `abusive.csv`/`new_kamusalay.csv`,
satu LogReg per label, `class_weight="balanced"`, model sub-label bersyarat pada `HS=1`).
`C=5` (terbaik di v1; selisih antar `C` kecil, jadi pencarian `C` dibuang agar CV cepat).

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

abusive_words = set(pd.read_csv(f"{RESOURCE_FOLDER}/abusive.csv")["ABUSIVE"].str.lower())
slang = pd.read_csv(f"{RESOURCE_FOLDER}/new_kamusalay.csv", header=None, names=["alay", "baku"],
                    encoding="latin-1")
SLANG_TO_FORMAL = dict(zip(slang["alay"], slang["baku"]))


def normalize_words(text):
    # huruf kecil, buang sisa byte emoji / \n literal / tanda baca, rapikan huruf berulang
    text = re.sub(r"\\x[0-9a-f]{2}", " ", str(text).lower()).replace("\\n", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    return [SLANG_TO_FORMAL.get(word, word) for word in text.split()]


def lexicon_features(texts):
    rows = []
    for text in texts:
        words = normalize_words(text)
        n_abusive = sum(word in abusive_words for word in words)
        rows.append([n_abusive, n_abusive > 0, n_abusive / max(1, len(words)), len(words),
                     str(text).count("USER"), str(text).isupper()])
    return np.array(rows, dtype=float)


def make_tfidf_model(C=5):                                    # UBAH: C
    features = FeatureUnion([
        ("kata", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("huruf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
        ("leksikon", make_pipeline(FunctionTransformer(lexicon_features), StandardScaler()))])
    classifier = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
    return make_pipeline(features, OneVsRestClassifier(classifier, n_jobs=-1))


oof_tfidf, test_tfidf = run_cv("tfidf", make_tfidf_model, train[TEXT].to_numpy(), test[TEXT].to_numpy())
print(report(Y, (oof_tfidf >= tune_thresholds(Y, oof_tfidf)).astype(int)).T.to_string())

[tfidf] fold 1/5: F1 val (ambang 0,5) 0.7079 (17 dtk)
[tfidf] fold 2/5: F1 val (ambang 0,5) 0.7057 (8 dtk)
[tfidf] fold 3/5: F1 val (ambang 0,5) 0.6988 (9 dtk)
[tfidf] fold 4/5: F1 val (ambang 0,5) 0.6944 (9 dtk)
[tfidf] fold 5/5: F1 val (ambang 0,5) 0.6709 (9 dtk)
[tfidf] F1 macro OOF (ambang per label): 0.7169 | tersimpan di outputs_cv/
                       HS    Abusive  HS_Individual   HS_Group  HS_Religion   HS_Race  HS_Physical  HS_Gender  HS_Other    HS_Weak  HS_Moderate  HS_Strong
jumlah_positif  4447.0000  4034.0000      2866.0000  1581.0000     636.0000  446.0000     262.0000   247.0000   2993.00  2710.0000    1366.0000   371.0000
diprediksi      4416.0000  4009.0000      3267.0000  1542.0000     724.0000  461.0000     257.0000   261.0000   3097.00  3488.0000    1611.0000   405.0000
F1                 0.8428     0.9096         0.7246     0.6558       0.6853    0.7674       0.5703     0.6024      0.78     0.7012       0.5805     0.7835


## 5. Unduh IndoBERTweet (sekali saja)
Model: [`indolem/indobertweet-base-uncased`](https://huggingface.co/indolem/indobertweet-base-uncased)
(Koto dkk., EMNLP 2021), IndoBERT yang vocabulary-nya diperluas dan dilatih lanjut pada Twitter
Indonesia. Disimpan di `pretrained/indobertweet`. Hanya 6 file yang diunduh (±450 MB); `optimizer.pt`
(890 MB, milik proses pelatihan aslinya) sengaja dilewati. Butuh internet; model publik, tidak perlu login.
Kalau sudah ada, sel ini tidak mengunduh lagi.

Kalau unduhan gagal (mis. jaringan kampus/proxy), unduh manual dari tab *Files* di halaman model:
`config.json`, `pytorch_model.bin`, `vocab.txt`, `tokenizer_config.json`, `special_tokens_map.json`,
`added_tokens.json`, lalu taruh di folder `pretrained/indobertweet`. Alternatif lewat terminal:
`hf download indolem/indobertweet-base-uncased --local-dir pretrained/indobertweet --exclude "optimizer.pt" "scheduler.pt" "training_args.bin"`.

In [6]:
# [DOWNLOAD] unduh IndoBERTweet ke pretrained/indobertweet
if "indobertweet" in BERT_MODELS:
    from huggingface_hub import snapshot_download

    INDOBERTWEET_FOLDER = Path(PRETRAINED_FOLDER) / "indobertweet"
    if (INDOBERTWEET_FOLDER / "pytorch_model.bin").exists():
        print("IndoBERTweet sudah ada di", INDOBERTWEET_FOLDER)
    else:
        snapshot_download(
            repo_id="indolem/indobertweet-base-uncased", local_dir=str(INDOBERTWEET_FOLDER),
            allow_patterns=["config.json", "pytorch_model.bin", "vocab.txt", "tokenizer_config.json",
                            "special_tokens_map.json", "added_tokens.json"])
        print("selesai:", sorted(p.name for p in INDOBERTWEET_FOLDER.iterdir() if p.is_file()))

IndoBERTweet sudah ada di C:\INAH-MINE\GEMASTIK-XIX\pretrained\indobertweet


C:\INAH-MINE\GEMASTIK-XIX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6. (Opsional) Vektor kalimat MiniLM + LogisticRegression, 5-fold
Default mati (`RUN_EMBEDDING=False`): di v1 skornya jauh di bawah TF-IDF/BERT dan bobot gabungannya kecil.
`neural_text` merapikan teks untuk model bahasa: huruf kecil, `USER` berulang jadi satu `@user`, sisa
byte emoji dibuang. `url_token` = kata pengganti `URL`: `tautan` untuk IndoBERT/MiniLM, `httpurl`
untuk IndoBERTweet (sesuai preprocessing di kartu modelnya: huruf kecil, `@USER`, `HTTPURL`).

In [7]:
def neural_text(text, url_token="tautan"):
    text = re.sub(r"\\x[0-9a-f]{2}", " ", str(text).lower()).replace("\\n", " ")
    text = re.sub(r"(?:\buser\b\s*)+", "@user ", text)
    text = re.sub(r"\burl\b", url_token, text)
    text = re.sub(r"\brt\b", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip(" '\"")


print(train[TEXT].head(3).map(neural_text).to_frame("neural").to_string(max_colwidth=90))

if RUN_EMBEDDING:
    from sentence_transformers import SentenceTransformer

    cache_train, cache_test = f"{CV_FOLDER}/emb_train.npy", f"{CV_FOLDER}/emb_test.npy"
    if os.path.exists(cache_train) and os.path.exists(cache_test):
        X_emb, X_emb_test = np.load(cache_train), np.load(cache_test)
    else:
        encoder = SentenceTransformer(model_path("minilm"), device=DEVICE)   # UBAH: minilm_en
        X_emb = encoder.encode(train[TEXT].map(neural_text).tolist(), batch_size=64,
                               normalize_embeddings=True, show_progress_bar=True)
        X_emb_test = encoder.encode(test[TEXT].map(neural_text).tolist(), batch_size=64,
                                    normalize_embeddings=True)
        np.save(cache_train, X_emb)
        np.save(cache_test, X_emb_test)
    print(X_emb.shape, X_emb_test.shape)

    def make_embedding_model(C=5):
        return OneVsRestClassifier(LogisticRegression(C=C, max_iter=3000, class_weight="balanced"), n_jobs=-1)

    oof_emb, test_emb = run_cv("embedding", make_embedding_model, X_emb, X_emb_test)

                                                                                      neural
0  - disaat semua cowok berusaha melacak perhatian gue. loe lantas remehkan perhatian yg ...
1  @user : @user siapa yang telat ngasih tau elu?edan sarap gue bergaul dengan cigax jifl...
2  41. kadang aku berfikir, kenapa aku tetap percaya pada tuhan padahal aku selalu jatuh ...


## 7a. Fine-tune BERT multi-label: dataset + fungsi latih per fold
Dari `finetune_bert.ipynb`, diadaptasi ke multi-label: 12 logit, `BCEWithLogitsLoss` + `sigmoid`,
`labels` = vektor float 12 angka, `pos_weight` (akar rasio negatif/positif, maks 5) untuk label jarang.
Pengaturan tetap dari template (lr 3e-5, batch 16, warmup 10% lalu turun linear, bf16).

Beda dari v1: tiap fold melatih model baru dari awal; **tanpa early stopping**, ambil epoch terakhir
(`SELECT_BEST_EPOCH=True` untuk memilih epoch terbaik di fold, tapi OOF jadi sedikit optimistis).
Di v1, F1 val masih naik sampai epoch 4 (0,689 → 0,727 → 0,757 → 0,762), jadi `EPOCHS=4` dipertahankan;
boleh dicoba 5. "CUDA out of memory": turunkan `BATCH_SIZE` ke 8.

In [8]:
# [BERT] definisi: lewati sel 7a-7b kalau tidak ada GPU (RUN_BERT=False)
import gc

from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

BACKBONES = {                                                 # UBAH: tambah backbone lain di sini
    "indobert": {"folder": "indobert", "url_token": "tautan", "lr": 3e-5},
    "indobertweet": {"folder": "indobertweet", "url_token": "httpurl", "lr": 3e-5},
}
MAX_LENGTH = 128                   # UBAH
BATCH_SIZE = 16                    # UBAH
EPOCHS = 4                         # UBAH: 3-5
SEEDS = [42]                       # UBAH: [42, 7] -> tiap fold dilatih 2x dan dirata-rata (2x lebih lama)
USE_POS_WEIGHT = True
USE_BF16 = DEVICE == "cuda"        # UBAH: False kalau GPU lama / error bf16
SELECT_BEST_EPOCH = False
FORCE_RETRAIN = False


class TextDataset(Dataset):
    def __init__(self, tokens, targets, indices):
        self.tokens, self.targets, self.indices = tokens, targets, np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        return {"input_ids": self.tokens["input_ids"][j],
                "attention_mask": self.tokens["attention_mask"][j],
                "labels": self.targets[j].tolist()}


def run_epoch(network, loader, loss_function, optimizer=None, scheduler=None):
    training = optimizer is not None
    network.train(training)
    total_loss, probabilities = 0.0, []
    for batch in tqdm(loader, leave=False):
        batch = batch.to(DEVICE)
        targets = batch.pop("labels")
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                logits = network(**batch).logits.float()
            loss = loss_function(logits, targets)
        if training:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        total_loss += loss.item() * len(targets)
        probabilities.append(torch.sigmoid(logits.detach()).cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(probabilities)


def train_fold(name, tokens, tokens_test, collate, index_train, index_val, seed):
    config = BACKBONES[name]
    torch.manual_seed(seed)
    np.random.seed(seed)
    network = AutoModelForSequenceClassification.from_pretrained(
        model_path(config["folder"]), num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
        problem_type="multi_label_classification").to(DEVICE)
    targets = Y.astype(np.float32)
    positives = Y[index_train].sum(axis=0)
    ratio = np.sqrt((len(index_train) - positives) / np.maximum(positives, 1))
    pos_weight = torch.tensor(np.clip(ratio, 1.0, 5.0), dtype=torch.float).to(DEVICE)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight if USE_POS_WEIGHT else None)
    optimizer = torch.optim.AdamW(network.parameters(), lr=config["lr"], weight_decay=0.01)
    train_loader = DataLoader(TextDataset(tokens, targets, index_train), batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate)
    val_loader = DataLoader(TextDataset(tokens, targets, index_val), batch_size=BATCH_SIZE * 2,
                            collate_fn=collate)
    test_data = TextDataset(tokens_test, np.zeros((len(test), len(LABELS)), dtype=np.float32),
                            np.arange(len(test)))                       # label pengisi
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE * 2, collate_fn=collate)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    best_f1, best_state = -1.0, None
    for epoch in range(EPOCHS):
        train_loss, _ = run_epoch(network, train_loader, loss_function, optimizer, scheduler)
        val_loss, val_proba = run_epoch(network, val_loader, loss_function)
        val_f1 = macro_f1(Y[index_val], (val_proba >= 0.5).astype(int))
        print(f"  epoch {epoch + 1}: train {train_loss:.3f} | val {val_loss:.3f} | F1 {val_f1:.4f}")
        if SELECT_BEST_EPOCH and val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in network.state_dict().items()}
    if best_state is not None:
        network.load_state_dict(best_state)
    _, val_proba = run_epoch(network, val_loader, loss_function)
    _, test_proba = run_epoch(network, test_loader, loss_function)
    del network, optimizer, scheduler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return val_proba, test_proba


def run_bert_cv(name):
    config = BACKBONES[name]
    tokenizer = AutoTokenizer.from_pretrained(model_path(config["folder"]))
    texts = train[TEXT].map(lambda t: neural_text(t, config["url_token"])).tolist()
    texts_test = test[TEXT].map(lambda t: neural_text(t, config["url_token"])).tolist()
    tokens = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
    tokens_test = tokenizer(texts_test, truncation=True, max_length=MAX_LENGTH)
    lengths = [len(ids) for ids in tokens["input_ids"][:2000]]
    print(f"[{name}] token per teks, persentil 50/95/99:", np.percentile(lengths, [50, 95, 99]))
    collate = DataCollatorWithPadding(tokenizer)
    return run_cv_neural(
        name, lambda index_train, index_val, seed: train_fold(
            name, tokens, tokens_test, collate, index_train, index_val, seed),
        seeds=SEEDS, force=FORCE_RETRAIN)

## 7b. Jalankan BERT 5-fold untuk tiap backbone di `BERT_MODELS`
Hasil per fold di-cache di `outputs_cv/`; kalau terhenti, jalankan ulang sel ini.

In [9]:
# [BERT] jalankan
if RUN_BERT:
    for backbone in BERT_MODELS:
        run_bert_cv(backbone)

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `5`.


[indobert] token per teks, persentil 50/95/99: [24. 56. 69.]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42103.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: C:\INAH-MINE\GEMASTIK-XIX\pretrained\indobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1: train 0.459 | val 0.304 | F1 0.6889


  epoch 2: train 0.262 | val 0.272 | F1 0.7274


  epoch 3: train 0.159 | val 0.259 | F1 0.7569


... (62 outputs left out)

  epoch 2: train 0.255 | val 0.280 | F1 0.7058


  epoch 3: train 0.168 | val 0.291 | F1 0.7320


  epoch 4: train 0.116 | val 0.291 | F1 0.7414


[indobertweet] fold 5 seed 42: selesai (6.0 menit)
[indobertweet] fold 5/5: F1 val (ambang 0,5) 0.7414
[indobertweet] F1 macro OOF (ambang per label): 0.7571 | tersimpan di outputs_cv/


## 8. Gabung semua model yang ada → `submission.csv`
1. Muat `outputs_cv/oof_<nama>.npy` dan `test_<nama>.npy` yang tersedia (tfidf, embedding, indobert, indobertweet).
2. Cari bobot rata-rata probabilitas (kelipatan 0,1) dengan F1 macro OOF terbaik (ambang per label disetel untuk tiap kandidat).
3. Setel ambang per label pada OOF gabungan, lalu bandingkan dekode **dengan** dan **tanpa** aturan hierarki; pakai yang lebih baik di OOF.
4. Terapkan bobot, ambang, dan dekode yang sama ke probabilitas test (rata-rata fold) → `submission.csv`.
Stabilitas: F1 per fold ditampilkan (rata-rata ± simpangan baku) sebagai gambaran derau.

In [10]:
import itertools

CANDIDATES = ("tfidf", "embedding", "indobert", "indobertweet")
names = [n for n in CANDIDATES
         if os.path.exists(f"{CV_FOLDER}/oof_{n}.npy") and os.path.exists(f"{CV_FOLDER}/test_{n}.npy")]
oof_probas = {n: np.load(f"{CV_FOLDER}/oof_{n}.npy") for n in names}
test_probas = {n: np.load(f"{CV_FOLDER}/test_{n}.npy") for n in names}
print("model tersedia:", names)


def weight_grid(n_models, step=0.1):
    total = int(round(1 / step))
    for parts in itertools.product(range(total + 1), repeat=n_models):
        if sum(parts) == total:
            yield np.array(parts) / total


def blend(probas, weights):
    return sum(w * probas[n] for w, n in zip(weights, names))


best_weights, best_score = None, -1.0
for weights in weight_grid(len(names)):
    mixed = blend(oof_probas, weights)
    score = macro_f1(Y, (mixed >= tune_thresholds(Y, mixed)).astype(int))
    if score > best_score:
        best_weights, best_score = weights, score
print("bobot terbaik:", {n: float(w) for n, w in zip(names, best_weights)}, "| F1 macro OOF:", round(best_score, 4))

mixed_oof = blend(oof_probas, best_weights)
thresholds = tune_thresholds(Y, mixed_oof)
plain_pred = (mixed_oof >= thresholds).astype(int)
hier_pred = decode_hierarchy(mixed_oof, thresholds)
plain_score, hier_score = macro_f1(Y, plain_pred), macro_f1(Y, hier_pred)
USE_HIERARCHY = hier_score >= plain_score
print(f"OOF tanpa hierarki {plain_score:.4f} | dengan hierarki {hier_score:.4f} -> pakai hierarki: {USE_HIERARCHY}")
oof_pred = hier_pred if USE_HIERARCHY else plain_pred
fold_scores = [macro_f1(Y[va], oof_pred[va]) for _, va in FOLDS]
print("F1 per fold:", np.round(fold_scores, 4), f"| rata-rata {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(pd.Series(thresholds, index=LABELS).round(2).to_string())

model tersedia: ['tfidf', 'indobert', 'indobertweet']
bobot terbaik: {'tfidf': 0.3, 'indobert': 0.3, 'indobertweet': 0.4} | F1 macro OOF: 0.7782
OOF tanpa hierarki 0.7782 | dengan hierarki 0.7761 -> pakai hierarki: False
F1 per fold: [0.7989 0.7706 0.7792 0.7747 0.7684] | rata-rata 0.7784 ± 0.0109
HS               0.45
Abusive          0.60
HS_Individual    0.40
HS_Group         0.50
HS_Religion      0.50
HS_Race          0.50
HS_Physical      0.40
HS_Gender        0.50
HS_Other         0.45
HS_Weak          0.40
HS_Moderate      0.45
HS_Strong        0.55


In [11]:
mixed_test = blend(test_probas, best_weights)
test_binary = decode_hierarchy(mixed_test, thresholds) if USE_HIERARCHY else (mixed_test >= thresholds).astype(int)

submission = pd.DataFrame(test_binary, columns=LABELS)
submission.insert(0, "id", test["id"].to_numpy())                           # UBAH
assert list(submission.columns) == list(sample_submission.columns)
submission.to_csv("submission.csv", index=False)
pd.DataFrame(mixed_test, columns=LABELS).assign(id=test["id"].to_numpy()).to_csv(
    f"{CV_FOLDER}/test_proba_ensemble.csv", index=False)
print(submission.shape)
print(submission[LABELS].mean().round(3).to_string())

(2636, 13)
HS               0.434
Abusive          0.372
HS_Individual    0.301
HS_Group         0.148
HS_Religion      0.063
HS_Race          0.049
HS_Physical      0.022
HS_Gender        0.016
HS_Other         0.310
HS_Weak          0.289
HS_Moderate      0.143
HS_Strong        0.040


## 9. Cek ke kunci jawaban (hanya demo; saat lomba tidak ada)
Ambang dan bobot ditentukan hanya dari OOF, jadi skor test ini jujur. Tabel kedua membandingkan tiap
model sendirian (ambang per label dari OOF) dengan gabungan. Skor v1 sebagai pembanding: gabungan 0,7519.
Dengan ±60 positif untuk label jarang, selisih < 0,01 masih dalam derau; jangan menyetel ulang
berdasarkan skor test.

In [12]:
Y_test = test_labels[LABELS].to_numpy().astype(int)
print("urutan id sama:", (test_labels["id"].to_numpy() == submission["id"].to_numpy()).all())
print("F1 macro test (gabungan):", round(macro_f1(Y_test, submission[LABELS].to_numpy()), 4),
      "| exact match:", round((Y_test == submission[LABELS].to_numpy()).all(axis=1).mean(), 4))
print(report(Y_test, submission[LABELS].to_numpy()).to_string())

rows = []
for n in names:
    thr_n = tune_thresholds(Y, oof_probas[n])
    rows.append({"model": n, "F1 OOF": macro_f1(Y, (oof_probas[n] >= thr_n).astype(int)),
                 "F1 test": macro_f1(Y_test, (test_probas[n] >= thr_n).astype(int))})
rows.append({"model": "gabungan", "F1 OOF": hier_score if USE_HIERARCHY else plain_score,
             "F1 test": macro_f1(Y_test, submission[LABELS].to_numpy())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

urutan id sama: True
F1 macro test (gabungan): 0.7798 | exact match: 0.7375
               jumlah_positif  diprediksi      F1
HS                       1114        1143  0.8959
Abusive                  1009         980  0.9291
HS_Individual             709         793  0.8003
HS_Group                  405         391  0.7688
HS_Religion               157         167  0.8025
HS_Race                   120         130  0.8240
HS_Physical                61          58  0.5714
HS_Gender                  59          42  0.6337
HS_Other                  747         816  0.8343
HS_Weak                   673         762  0.7847
HS_Moderate               339         378  0.7057
HS_Strong                 102         106  0.8077
       model  F1 OOF  F1 test
       tfidf  0.7169   0.7301
    indobert  0.7477   0.7578
indobertweet  0.7571   0.7594
    gabungan  0.7782   0.7798
